# BirdVision-AI : fine-tuning de YOLO26s sur les oiseaux

Objectif : un YOLO26s spécialisé **oiseaux** (une seule classe `bird`), plus confiant
et meilleur sur les petits oiseaux et les oiseaux en vol.

**Avant de commencer :** menu *Exécution → Modifier le type d'exécution → GPU T4*, puis lancer les cellules dans l'ordre.

Durée estimée : 15 min de préparation + environ 2 h d'entraînement.
Les résultats sont enregistrés dans Google Drive : si Colab se déconnecte, relance simplement
la cellule 5, l'entraînement reprend là où il s'était arrêté.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

## 2. Installer les bibliothèques

In [ ]:
!pip install -q ultralytics==8.4.158 fiftyone

## 3. Connecter Google Drive (pour sauvegarder le modèle)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/BirdVision-AI'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

## 4. Télécharger un jeu de données d'oiseaux (Open Images V7)

Images réelles avec boîtes « Bird » annotées à la main : oiseaux posés, en vol, en groupe, petits et grands.
Gratuit, pas de compte nécessaire.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

N_TRAIN = 4000   # augmenter (ex : 8000) si tu as le temps, pour un meilleur modèle
N_VAL = 600
EXPORT_DIR = '/content/birds_yolo'

for split, n in [('train', N_TRAIN), ('validation', N_VAL)]:
    ds = foz.load_zoo_dataset(
        'open-images-v7',
        split=split,
        label_types=['detections'],
        classes=['Bird'],
        max_samples=n,
        shuffle=True,
        seed=42,
        dataset_name=f'birds-{split}',
    )
    view = ds.filter_labels('ground_truth', F('label') == 'Bird')
    view.export(
        export_dir=EXPORT_DIR,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field='ground_truth',
        split='train' if split == 'train' else 'val',
        classes=['Bird'],
    )

# Nom de classe en minuscules : le backend BirdVision filtre sur "bird"
yaml_path = f'{EXPORT_DIR}/dataset.yaml'
text = open(yaml_path).read().replace('Bird', 'bird')
open(yaml_path, 'w').write(text)
print(text)

## 5. Entraîner YOLO26s

On part du YOLO26s officiel (celui de BirdVision) et on le spécialise sur les oiseaux.
`imgsz=1280` comme dans le backend, pour bien voir les petits oiseaux.

In [ ]:
import os
from ultralytics import YOLO

PROJECT = f'{DRIVE_DIR}/runs'
LAST = f'{PROJECT}/yolo26s_birds/weights/last.pt'

if os.path.exists(LAST):
    print('Reprise de l\'entraînement interrompu...')
    model = YOLO(LAST)
    model.train(resume=True)
else:
    model = YOLO('yolo26s.pt')
    model.train(
        data=f'{EXPORT_DIR}/dataset.yaml',
        epochs=40,
        patience=10,        # arrêt anticipé si plus de progrès
        imgsz=1280,
        batch=8,            # si erreur "CUDA out of memory" : mettre 4
        project=PROJECT,
        name='yolo26s_birds',
        exist_ok=True,
        save_period=1,      # sauvegarde à chaque époque (utile si Colab coupe)
        plots=True,
    )

## 6. Évaluer le modèle

In [ ]:
best = YOLO(f'{PROJECT}/yolo26s_birds/weights/best.pt')
metrics = best.val(data=f'{EXPORT_DIR}/dataset.yaml', imgsz=1280)
print(f"mAP50     : {metrics.box.map50:.3f}")
print(f"mAP50-95  : {metrics.box.map:.3f}")
print(f"Précision : {metrics.box.mp:.3f}")
print(f"Rappel    : {metrics.box.mr:.3f}")

## 7. Tester sur tes propres photos

Clique sur l'icône dossier à gauche, glisse tes photos (ex : `oiseau2.jpg`) dans `/content`, puis lance la cellule.

In [ ]:
import glob
from IPython.display import Image, display

photos = [p for p in glob.glob('/content/*.jpg') + glob.glob('/content/*.png')]
for p in photos:
    r = best.predict(p, imgsz=1280, conf=0.25, save=True, project='/content/tests', name='pred', exist_ok=True)[0]
    confs = sorted(r.boxes.conf.tolist(), reverse=True)
    print(f"{os.path.basename(p)} : {len(confs)} oiseaux | conf : {', '.join(f'{c:.2f}' for c in confs)}")
    display(Image(f'/content/tests/pred/{os.path.basename(p)}', width=700))

## 8. Télécharger le modèle

Le fichier est aussi dans Google Drive : `MyDrive/BirdVision-AI/yolo26s_birds.pt`.
Place-le ensuite dans `BirdVision-AI\models\` sur ton PC.

In [ ]:
import shutil
from google.colab import files

final = f'{DRIVE_DIR}/yolo26s_birds.pt'
shutil.copy(f'{PROJECT}/yolo26s_birds/weights/best.pt', final)
files.download(final)